**Environmnet Preparation**
- Creating New volume for holding 5 CSV files of Customers
- Loading 5 csv files from local PC into Unity volume
- Incrementaly load data with using of leagacy COPY INTO

In [0]:
CREATE VOLUME data_fusion.source_data.incremental_ingestion

In [0]:
%python

# Renaming old folder name, this is move operation

old_folder = '/Volumes/data_fusion/source_data/incremental_ingestion/CSV_Fiels_batch'
new_folder = '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_CSV'

dbutils.fs.mv(old_folder,new_folder,recurse=True)

In [0]:
%python

# Creating new directory where I will be gradually move files from Orders_CSV
 
dbutils.fs.mkdirs('/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV')

In [0]:
%python
folder_root = '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_CSV/'

# loading all Path to files into List Variable

files_list = [file.path for file in dbutils.fs.ls(folder_root)]

# Moving first 2 CSV files to Orders_incrementaly_loaded
target_location = '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV/'

for path in files_list[:2]:
    dbutils.fs.cp(path,target_location)

In [0]:
-- Table creation for incremental load
-- table is created with no schema
DROP TABLE IF EXISTS data_fusion.bronze_delta_tables.orders_incremental_load;
CREATE TABLE data_fusion.bronze_delta_tables.orders_incremental_load;

**COPY INTO**
- is idempotent (load data only from new files)
- is available only for Spark SQL (not for python)
- Is Lgeacy (now replaced by Autoloader)
- If I delete any data from DElata table this will not be reloaded by COPY_INTO

In [0]:
COPY INTO data_fusion.bronze_delta_tables.orders_incremental_load
FROM '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV/'
FILEFORMAT = csv
COPY_OPTIONS ("mergeschema" = "true")

In [0]:
%python
# Moving 3rd and 4th CSV file to Orders_incrementaly_loaded
target_location = '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV/'

for path in files_list[2:4]:
    dbutils.fs.cp(path,target_location)

In [0]:
COPY INTO data_fusion.bronze_delta_tables.orders_incremental_load
FROM '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV/'
FILEFORMAT = csv
COPY_OPTIONS ("mergeschema" = "true")

In [0]:
%python
# Moving 5th CSV file to Orders_incrementaly_loaded
target_location = '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV/'

for path in files_list[4:]:
    dbutils.fs.cp(path,target_location)

In [0]:
COPY INTO data_fusion.bronze_delta_tables.orders_incremental_load
FROM '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV/'
FILEFORMAT = csv
COPY_OPTIONS ("mergeschema" = "true")

In [0]:
%python

# removeing all files from Orders_incrementaly_loaded_CSV

folder_path_target = '/Volumes/data_fusion/source_data/incremental_ingestion/Orders_incrementaly_loaded_CSV/'

for directory in dbutils.fs.ls(folder_path_target):
    dbutils.fs.rm(directory.path)